# Assignment 1.2: RAG-Based Intelligent Document Assistant



In [ ]:
# Assignment 1.2: RAG-Based Intelligent Document Assistant
# Google Colab-ready notebook script
# Stack: LlamaIndex (indexing/retrieval) + LangChain (LLM/prompt/chain/output parser)

## Install dependencies

In [ ]:
%pip install requests==2.32.4

In [ ]:
import requests
import google.colab

print("request version:",requests.__version__)
print("Google Colab imported successfully")

request version: 2.32.4
Google Colab imported successfully


In [ ]:
import huggingface_hub

print("Hugging Face Hub version:", huggingface_hub.__version__)

Hugging Face Hub version: 1.33.0


In [ ]:
%pip install requests==2.34.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%pip install -q -U llama-index-core llama-index-embeddings-huggingface langchain-core langchain-huggingface huggingface_hub sentence-transformers pypdf requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 13.5 MB/s eta 0:00:00


## Imports

In [ ]:
import os
import time
import requests

from getpass import getpass
from google.colab import files
from pypdf import PdfReader
from bs4 import BeautifulSoup

from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint

## Configure embedding model and text chunking

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")

    if not hf_token:
        raise ValueError("HF_TOKEN is empty. Check Colab Secrets.")

    login(token=hf_token)

    print("Hugging Face authentication successful!")

except Exception as e:
    print("Authentication error:", e)

Hugging Face authentication successful!


In [ ]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
Settings.text_splitter = SentenceSplitter(
    chunk_size=700,
    chunk_overlap=100
)
print("Embedding model configured successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model configured successfully.


## Upload PDF/TXT files

In [ ]:
uploaded_files = files.upload()
print("\nUploaded files:")
for filename in uploaded_files:
    print("-", filename)

Saving MachineLearning.pdf to MachineLearning.pdf
Saving RAG AI.pdf to RAG AI.pdf

Uploaded files:
- MachineLearning.pdf
- RAG AI.pdf


## Extract text from uploaded documents

In [ ]:
documents = []
failed_files = []

for filename in uploaded_files.keys():
    try:
        if filename.lower().endswith(".pdf"):
            reader = PdfReader(filename)
            text_content = "\n".join(
                page.extract_text() or "" for page in reader.pages
            )
        elif filename.lower().endswith(".txt"):
            with open(filename, "r", encoding="utf-8") as f:
                text_content = f.read()
        else:
            print(f"Skipped unsupported file: {filename}")
            continue

        if not text_content.strip():
            print(f"No extractable text found: {filename}")
            failed_files.append(filename)
            continue

        documents.append(
            Document(
                text=text_content,
                metadata={"file_name": filename}
            )
        )
        print(f"Processed: {filename}")

    except Exception as exc:
        failed_files.append(filename)
        print(f"Could not process {filename}: {exc}")

if not documents:
    raise ValueError(
        "No readable documents found. Upload text-based PDF or TXT files."
    )

print(f"\nSuccessfully loaded {len(documents)} documents.")
if failed_files:
    print("Failed or empty files:", failed_files)
print("Note: scanned/image-only PDFs require OCR.")

Processed: MachineLearning.pdf
Processed: RAG AI.pdf

Successfully loaded 2 documents.
Note: scanned/image-only PDFs require OCR.


## Optional webpage ingestion (press Enter to skip)

In [ ]:
url = input("Enter a public webpage URL (or press Enter to skip): ").strip()

if url:
    try:
        response = requests.get(
            url,
            timeout=20,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        webpage_text = soup.get_text(separator=" ", strip=True)
        if webpage_text:
            documents.append(
                Document(
                    text=webpage_text,
                    metadata={"file_name": url}
                )
            )
            print("Webpage added successfully.")
        else:
            print("No readable text found on this webpage.")
    except Exception as exc:
        print("Could not load webpage:", exc)
else:
    print("Skipping web ingestion.")

Enter a public webpage URL (or press Enter to skip): 
Skipping web ingestion.


## Build LlamaIndex vector index and retriever

In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
    transformations=[Settings.text_splitter],
    show_progress=True
)
retriever = index.as_retriever(similarity_top_k=4)
print("LlamaIndex vector index created successfully.")
print("Number of source documents:", len(documents))

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/120 [00:00<?, ?it/s]

LlamaIndex vector index created successfully.
Number of source documents: 2


## Configure Hugging Face token and LangChain LLM

In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

client = InferenceClient(
    provider="auto",
    api_key=hf_token
)

In [ ]:
# Create/access a token at https://huggingface.co/settings/tokens.
# Keep the token private. The model must be available via an enabled Inference Provider.
hf_token = getpass("Enter your Hugging Face access token: ").strip()
if not hf_token:
    raise ValueError("A Hugging Face token is required.")

# If this model is unavailable to your account/provider, change repo_id to
# a currently supported instruction model available through your HF Inference Provider.
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=400,
    do_sample=False,
    huggingfacehub_api_token=hf_token
)
print("LangChain language model configured.")

Enter your Hugging Face access token: ··········
LangChain language model configured.


In [ ]:
from google.colab import userdata
from huggingface_hub import InferenceClient

try:
    token = userdata.get("HF_TOKEN")
    print("HF token found:", bool(token))

    client = InferenceClient(
        provider="auto",
        api_key=token
    )

    result = client.chat_completion(
        messages=[
            {
                "role": "user",
                "content": "Explain machine learning in simple words."
            }
        ],
        model="openai/gpt-oss-120b",
        max_tokens=100
    )

    print(result.choices[0].message.content)

except Exception as e:
    print(type(e).__name__, str(e))

HF token found: True
HfHubHTTPError Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6aca4b28-25960f5b5bb612c876ad6a0a;c1701060-fd2c-490b-bbc4-28c6d0d049f7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 24 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (490 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    response = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=10
    )
    print("Ollama server status:", response.status_code)
except requests.RequestException as e:
    print("Server connection failed:", e)

Ollama server status: 200


In [ ]:
!ollama pull llama3.2:1b

In [ ]:
!ollama run llama3.2:1b "Explain RAG in simple words."

RAG is a term that stands for "Red, Amber, and Green". It's not a person's 
name, but rather a color code that's often used to categorize and visualize
visualize different levels of activity, severity, or importance.

Here's a simple breakdown:

- **Red**: This color typically represents something that's serious, urgent
urgent, or critical. It's like a "stop" sign – when something red, amber, a
and green is raised, it means something needs immediate attention or action
action.
- **Amber**: This color represents something that's important but not criti
critical. It's like a warning sign – when amber is raised, it means somethi
something is worth paying attention to, but it's not as urgent as red.
- **Green**: This color is often used for things that are normal, safe, or 
routine. It's like a "go" sign – when green is raised, it means everything 
is okay and nothing needs to be done.

In practical terms, RAG is often used in:

- Emergency services, like fire departments or ambulance serv

## Define LangChain RAG prompt and chain

In [ ]:
prompt = PromptTemplate.from_template("""
You are an intelligent document question-answering assistant.

Answer the question using ONLY the context below.

Rules:
1. Be accurate, clear, and concise.
2. Do not invent facts absent from the context.
3. If the context does not contain the answer, say:
   "The answer was not found in the provided documents."
4. Explain important points in simple language.

Context:
{context}

Question:
{question}

Answer:
""")

rag_chain = prompt | llm | StrOutputParser()
print("LangChain RAG chain created successfully.")

LangChain RAG chain created successfully.


## Question-answering function

In [ ]:
def ask_documents(question):
    if not isinstance(question, str) or not question.strip():
        print("Please enter a valid question.")
        return

    try:
        start_time = time.time()
        retrieved_nodes = retriever.retrieve(question)

        if not retrieved_nodes:
            print("No relevant passages were retrieved.")
            return

        context_parts = []
        sources = []

        for i, item in enumerate(retrieved_nodes, start=1):
            chunk_text = item.node.get_content()
            source = item.node.metadata.get("file_name", "Unknown source")
            sources.append(source)
            context_parts.append(f"[Source {i}: {source}]\n{chunk_text}")

        context = "\n\n".join(context_parts)
        answer = rag_chain.invoke({
            "context": context,
            "question": question
        })
        elapsed = time.time() - start_time

        print("\n" + "=" * 60)
        print("QUESTION")
        print(question)
        print("\nANSWER")
        print(answer)
        print("\nSOURCE DOCUMENTS")
        for source in dict.fromkeys(sources):
            print("-", source)
        print(f"\nRetrieved passages: {len(retrieved_nodes)}")
        print(f"Response time: {elapsed:.2f} seconds")
        print("=" * 60)

    except Exception as exc:
        print("\nQuestion-answering error:", str(exc))
        print(
            "Check the Hugging Face token, model/provider availability, "
            "internet connection, and runtime logs."
        )

## Ask one question

In [ ]:
import requests

def ask_documents(question):
    try:
        # Retrieve relevant document context
        docs = retriever.retrieve(question)

        context = "\n\n".join(
            doc.page_content for doc in docs
        )

        # Generate an answer using Ollama
        response = requests.post(
            "http://127.0.0.1:11434/api/chat",
            json={
                "model": "llama3.2:1b",
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            "Answer using the provided document context. "
                            "If the answer is not in the context, say so."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Document context:\n{context}\n\n"
                            f"Question: {question}"
                        )
                    }
                ],
                "stream": False
            },
            timeout=180
        )

        response.raise_for_status()
        print(response.json()["message"]["content"])

    except Exception as e:
        print("Question-answering error:", str(e))

In [ ]:
import subprocess
import time
import requests

# Check whether Ollama is already running
try:
    r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
    print("Ollama is already running:", r.status_code == 200)

except requests.RequestException:
    # Start the server if it is not running
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(8)

    try:
        r = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=5
        )
        print("Ollama server status:", r.status_code)
    except requests.RequestException as e:
        print("Ollama failed to start:", e)

Ollama is already running: True


In [ ]:
!apt-get update -qq
!apt-get install -y zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
curl is already the newest version (8.5.0-2ubuntu10.15).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os
import shutil
import subprocess

os.environ["PATH"] += ":/usr/local/bin:/usr/bin:/bin"

print("Ollama path:", shutil.which("ollama"))

result = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True
)
print(result.stdout or result.stderr)

Ollama path: /usr/local/bin/ollama
ollama version is 0.40.2



In [ ]:
!ollama pull llama3.2:1b

In [ ]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:1b    baf6a787fdff    1.3 GB    4 seconds ago    


In [ ]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    r = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=5
    )
    print(r.status_code)
    print(r.json())
except Exception as e:
    print("Server error:", e)

200
{'models': [{'name': 'llama3.2:1b', 'model': 'llama3.2:1b', 'modified_at': '2026-10-10T14:31:56.152015012Z', 'size': 1321098329, 'digest': 'baf6a787fdffd633537aa2eb51cfd54cb93ff08e28040095462bb63daf552878', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '1.2B', 'quantization_level': 'Q8_0', 'context_length': 131072, 'embedding_length': 2048, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools']}]}


In [ ]:
import os
import shutil
import subprocess

os.environ["PATH"] += ":/usr/local/bin:/usr/bin:/bin"

print("Ollama path:", shutil.which("ollama"))

subprocess.run(["ollama", "--version"], check=True)

Ollama path: /usr/local/bin/ollama


CompletedProcess(args=['ollama', '--version'], returncode=0)

In [ ]:
import requests

def ask_documents(question):
    try:
        # Retrieve relevant chunks using LlamaIndex
        docs = retriever.retrieve(question)

        # Extract text correctly from LlamaIndex results
        context = "\n\n".join(
            item.node.get_content()
            for item in docs
            if item.node.get_content().strip()
        )

        if not context.strip():
            print("No relevant content found in the indexed documents.")
            return

        # Generate answer using Ollama
        response = requests.post(
            "http://127.0.0.1:11434/api/chat",
            json={
                "model": "llama3.2:1b",
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            "Answer using the supplied document context. "
                            "If the answer is not in the context, "
                            "clearly say so."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Context:\n{context}\n\n"
                            f"Question: {question}"
                        )
                    }
                ],
                "stream": False
            },
            timeout=180
        )

        response.raise_for_status()
        print(response.json()["message"]["content"])

    except Exception as e:
        print("Question-answering error:", repr(e))

In [ ]:
question = input("Ask a question about your documents: ")
ask_documents(question)

Ask a question about your documents: What is Retrieval-Augmented Generation (RAG), and how does it improve the performance of large language models?
Retrieval-Augmented Generation (RAG) is a research area that aims to improve the performance of large language models (LLMs) by incorporating external knowledge or data in the generation process. In traditional LLMs, the model generates text based solely on the input prompt, without any external information. RAG modifies this approach by combining the generated text with external knowledge or data retrieved from external sources, such as databases or articles.

The primary goal of RAG is to enhance the accuracy and robustness of LLMs by leveraging the strengths of both the model and the external knowledge. By incorporating external information, RAG can:

1. Improve the generation of coherent and relevant text.
2. Enhance the model's ability to handle complex and nuanced questions.
3. Reduce the need for manual curation or editing of genera

## Optional interactive question loop

In [ ]:
# Run this cell when you want to ask multiple questions.
# Type exit, quit, or stop to end the session.
print("RAG Document Assistant is ready! Type 'exit' to stop.\n")
while True:
    question = input("Your question: ").strip()
    if question.lower() in {"exit", "quit", "stop"}:
        print("Session ended.")
        break
    ask_documents(question)

RAG Document Assistant is ready! Type 'exit' to stop.

Your question: EXIT
Session ended.


## Summary request

In [ ]:
# Run separately when you want a summary based on retrieved passages.
summary_question = (
    "Summarize the main topics, key concepts, important definitions, "
    "and major conclusions in the provided context. Use headings and bullets."
)
ask_documents(summary_question)

**Key Concepts and Definitions**

* **Graphical Representations**: Using charts such as histograms, box plots, scatter plots, and bar charts to visualize relationships within the data and distributions of variables.
* **Outlier Detection**: Identifying unusual values that deviate from other data points, which can influence statistical analyses and indicate data entry errors or unique cases.
* **Correlation Analysis**: Checking the relationships between variables to understand how they might affect each other, including computing correlation coefficients and creating correlation matrices.
* **Handling Missing Values**: Detecting and deciding how to address missing data points, whether by imputation or removal, depending on their impact and the amount of missing data.
* **Summary Statistics**: Calculating key statistics that provide insight into data trends and nuances.

**Main Topics and Key Concepts**

* **Machine Learning Study Material**: A collection of resources for machine learnin